In [21]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay 
from sklearn.model_selection import GridSearchCV

In [22]:
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

print(train.shape)
print(test.shape)

train.head()

(22869, 5)
(5718, 5)


,clean_text,type,queue,priority,language
0,integrating clickup saas platform customer sup...,Request,Technical Support,medium,en
1,issue with data access for healthcare provider...,Problem,Product Support,high,de
2,digital support inquiry sehr geehrter kundendi...,Request,Technical Support,low,de
3,möglichkeiten zur integration von jira bewährt...,Request,IT Support,high,de
4,inquiry about security capabilities in healthc...,Request,Billing and Payments,high,en


In [23]:
X_train = train["clean_text"]
y_train = train["priority"]

X_test = test["clean_text"]
y_test = test["priority"]

In [24]:
y_train.value_counts()

priority
medium    9158
high      8988
low       4723
Name: count, dtype: int64

## Logistic Regression Baseline


In [25]:
priority_lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

priority_lr_model.fit(X_train, y_train)

y_pred_lr = priority_lr_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.5874431619447359
              precision    recall  f1-score   support

        high       0.63      0.63      0.63      2190
         low       0.47      0.56      0.51      1171
      medium       0.61      0.56      0.59      2357

    accuracy                           0.59      5718
   macro avg       0.57      0.58      0.58      5718
weighted avg       0.59      0.59      0.59      5718



### LinearSVC Baseline

In [26]:
priority_svc_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("classifier", LinearSVC(
        class_weight="balanced"
    ))
])

priority_svc_model.fit(X_train, y_train)

y_pred_svc = priority_svc_model.predict(X_test)

print("LinearSVC Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))

LinearSVC Accuracy: 0.6346624693948933
              precision    recall  f1-score   support

        high       0.66      0.68      0.67      2190
         low       0.58      0.54      0.56      1171
      medium       0.64      0.64      0.64      2357

    accuracy                           0.63      5718
   macro avg       0.63      0.62      0.62      5718
weighted avg       0.63      0.63      0.63      5718



### Improved LinearSVC With Word + Character Features

In [27]:
linear_svc_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            max_features=50000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            max_features=30000,
            min_df=2,
            sublinear_tf=True
        ))
    ])),
    ("classifier", LinearSVC(
        class_weight="balanced",
        C=1.0,
        max_iter=5000
    ))
])

linear_svc_model.fit(X_train, y_train)

y_pred_svc = linear_svc_model.predict(X_test)

print("Improved LinearSVC Accuracy:", accuracy_score(y_test, y_pred_svc))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svc))

Improved LinearSVC Accuracy: 0.6409583770549143

Classification Report:
              precision    recall  f1-score   support

        high       0.67      0.69      0.68      2190
         low       0.57      0.56      0.57      1171
      medium       0.65      0.64      0.64      2357

    accuracy                           0.64      5718
   macro avg       0.63      0.63      0.63      5718
weighted avg       0.64      0.64      0.64      5718



### Hyperparameter Tuning With GridSearchCV

In [28]:
priority_svc_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        max_iter=7000
    ))
])

In [29]:
param_grid = {
    "tfidf__ngram_range": [(1, 1),(1, 2),(1, 3)],

    "tfidf__max_features": [30000,50000,70000],

    "tfidf__min_df": [1,2,3],

    "tfidf__max_df": [0.90,0.95],

    "classifier__C": [0.1,0.5,1,2,5]
}

In [30]:
grid_search = GridSearchCV(
    estimator=priority_svc_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 270 candidates, totalling 1350 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=7000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 0.5, ...], 'tfidf__max_df': [0.9, 0.95], 'tfidf__max_features': [30000, 50000, ...], 'tfidf__min_df': [1, 2, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time f

In [31]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Macro F1:")
print(grid_search.best_score_)

Best Parameters:
{'classifier__C': 5, 'tfidf__max_df': 0.9, 'tfidf__max_features': 70000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}

Best CV Macro F1:
0.6163168790656891


In [32]:
best_priority_model = grid_search.best_estimator_

y_pred_grid = best_priority_model.predict(X_test)

print("GridSearch LinearSVC Accuracy:", accuracy_score(y_test, y_pred_grid))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_grid))

GridSearch LinearSVC Accuracy: 0.6670164393144457

Classification Report:
              precision    recall  f1-score   support

        high       0.68      0.70      0.69      2190
         low       0.63      0.58      0.60      1171
      medium       0.67      0.68      0.67      2357

    accuracy                           0.67      5718
   macro avg       0.66      0.65      0.66      5718
weighted avg       0.67      0.67      0.67      5718



In [33]:
grid_report = classification_report(
    y_test,
    y_pred_grid,
    output_dict=True,
    zero_division=0
)

grid_report_df = pd.DataFrame(grid_report).transpose()

display(grid_report_df)

,precision,recall,f1-score,support
high,0.680672,0.702740,0.691530,2190.000000
low,0.630009,0.577284,0.602496,1171.000000
medium,0.670721,0.678405,0.674541,2357.000000
accuracy,0.667016,0.667016,0.667016,0.667016
macro avg,0.660468,0.652810,0.656189,5718.000000
weighted avg,0.666195,0.667016,0.666294,5718.000000


In [34]:
lr_report = classification_report(y_test, y_pred_lr, output_dict=True, zero_division=0)
svc_report = classification_report(y_test, y_pred_svc, output_dict=True, zero_division=0)
grid_report = classification_report(y_test, y_pred_grid, output_dict=True, zero_division=0)

# Model comparison table
model_comparison = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "LinearSVC",
        "GridSearch LinearSVC"
    ],
    "accuracy": [
        lr_report["accuracy"],
        svc_report["accuracy"],
        grid_report["accuracy"]
    ],
    "macro_f1": [
        lr_report["macro avg"]["f1-score"],
        svc_report["macro avg"]["f1-score"],
        grid_report["macro avg"]["f1-score"]
    ],
    "weighted_f1": [
        lr_report["weighted avg"]["f1-score"],
        svc_report["weighted avg"]["f1-score"],
        grid_report["weighted avg"]["f1-score"]
    ]
})

model_comparison = model_comparison.sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)

display(model_comparison)

model_comparison.to_csv(
    "../reports/priority_model_comparison.csv",
    index=False
)

,model,accuracy,macro_f1,weighted_f1
0,GridSearch LinearSVC,0.667016,0.656189,0.666294
1,LinearSVC,0.640958,0.628984,0.640543
2,Logistic Regression,0.587443,0.577435,0.588869


In [35]:
best_model_name = model_comparison.loc[0, "model"]

if best_model_name == "Logistic Regression":
    final_priority_model = logistic_model
    y_pred_best = y_pred_lr

elif best_model_name == "LinearSVC":
    final_priority_model = linear_svc_model
    y_pred_best = y_pred_svc

else:
    final_priority_model = grid_search.best_estimator_
    y_pred_best = y_pred_grid

print("Best model:", best_model_name)
print("Best macro F1:", model_comparison.loc[0, "macro_f1"])

Best model: GridSearch LinearSVC
Best macro F1: 0.6561889243193701


In [36]:
joblib.dump(
    final_priority_model,
    "../models/ticket_priority_baseline.pkl"
)

['../models/ticket_priority_baseline.pkl']

In [37]:
wrong_priority_predictions = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "actual_priority": y_test.reset_index(drop=True),
    "predicted_priority": pd.Series(y_pred_best)
})

wrong_priority_predictions = wrong_priority_predictions[
    wrong_priority_predictions["actual_priority"] !=
    wrong_priority_predictions["predicted_priority"]
]

wrong_priority_predictions.to_csv(
    "../reports/wrong_priority_predictions.csv",
    index=False
)

display(wrong_priority_predictions.head(20))

,text,actual_priority,predicted_priority
0,potential data breach in healthcare system the...,low,medium
1,hilfe zur entwicklung digitaler wachstumsmetho...,high,low
6,enhanced security implement advanced security ...,medium,high
10,probleme bei investmentsoptimierungsmodul das ...,low,high
12,request for update and optimization of data an...,medium,high
14,concern about billing discrepancies our market...,medium,low
16,"kontakten sie mich, um mehr über digitale stra...",medium,high
19,untersuchung datenverletzung erforderlich es g...,medium,high
24,änderung der rückgabeverordnungen lieber kunde...,medium,high
26,anfrage zu digitalen marketing-tools könnten s...,medium,low
